# Demonstração de uso do MariaDB

Simula um **client** comunicando com o banco de dados relacional pela rede.

In [ ]:
import os
import geopandas as gpd
import leafmap
import pandas as pd
from dotenv import load_dotenv
from shapely.geometry import shape
from sqlalchemy import create_engine, text


In [2]:
load_dotenv()

engine = create_engine(
    "mysql+pymysql://{user}:{pswd}@{host}:{port}/{db}".format(
        user=os.getenv("MARIADB_USER"),
        pswd=os.getenv("MARIADB_PASSWORD"),
        host=os.getenv("HOST"),
        port=os.getenv("PORT"),
        db=os.getenv("MARIADB_DATABASE"),
    )
)


## 1. Selecione uma área

Desenhe um polígono ou retângulo para filtrar os focos de fogo.

In [7]:
m = leafmap.Map(center=[-14, -52], zoom=4)
m.add_basemap("OpenStreetMap")
m


OpenStreetMap has been already added before.


Map(center=[-14, -52], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

## 2. Consulta os focos de fogo e faz os devidos JOINs 

Este código filtra e une as bases na sequência `focos_de_fogo` → `municipios` → `estados` pelas suas chaves secundárias (`municipio_id`, `estado_id`), utilizando `ST_Intersects` em suas geometrias.

In [ ]:
def query_focos_in_roi(m: leafmap.Map) -> gpd.GeoDataFrame:
    if m.user_roi is None:
        raise ValueError("Draw an area on the map above before running this query.")

    roi_wkt = shape(m.user_roi["geometry"]).wkt

    query = text(
        """
        SELECT
            f.id_foco_bdq,
            f.satelite,
            f.data_hora_gmt,
            f.frp,
            f.risco_fogo,
            f.bioma,
            m.nome AS municipio,
            e.nome AS estado,
            e.regiao,
            ST_AsBinary(f.geometry) AS geometry
        FROM focos_de_fogo f
        JOIN municipios m ON m.id = f.municipio_id
        JOIN estados e ON e.id = m.estado_id
        WHERE ST_Intersects(f.geometry, ST_GeomFromText(:roi_wkt, 4674))
        """
    )

    with engine.connect() as conn:
        df = pd.read_sql(query, con=conn, params={"roi_wkt": roi_wkt})

    return gpd.GeoDataFrame(
        df.drop(columns=["geometry"]),
        geometry=gpd.GeoSeries.from_wkb(df["geometry"]),
        crs="EPSG:4674",
    )


focos_in_roi = query_focos_in_roi(m)
print(f"{len(focos_in_roi)} focos de fogo encontrados na área de interesse.")
focos_in_roi.drop(columns="geometry")


60 focos de fogo found in the selected area


,id_foco_bdq,satelite,data_hora_gmt,frp,risco_fogo,bioma,municipio,estado,regiao
0,1830058903,GOES-19,2026-08-03 06:10:00,48.9,None,Cerrado,CRISTAIS,MINAS GERAIS,SUDESTE
1,1830066670,NOAA-20,2026-08-03 04:42:00,5.2,None,Cerrado,CRISTAIS,MINAS GERAIS,SUDESTE
2,1830058246,NOAA-21,2026-08-03 03:47:00,4.1,None,Cerrado,CRISTAIS,MINAS GERAIS,SUDESTE
3,1830070234,NOAA-20,2026-08-03 04:42:00,5.2,None,Cerrado,CRISTAIS,MINAS GERAIS,SUDESTE
4,1830058107,NOAA-21,2026-08-03 03:47:00,4.1,None,Cerrado,CRISTAIS,MINAS GERAIS,SUDESTE
5,1830179800,NOAA-21,2026-08-03 16:16:00,6.4,None,Mata Atlântica,TRÊS PONTAS,MINAS GERAIS,SUDESTE
6,1830058402,NOAA-21,2026-08-03 03:49:00,1.3,None,Mata Atlântica,PARAIBUNA,SÃO PAULO,SUDESTE
7,1830058696,NOAA-21,2026-08-03 03:49:00,1.3,None,Mata Atlântica,PARAIBUNA,SÃO PAULO,SUDESTE
8,1830058425,NOAA-21,2026-08-03 03:49:00,2.2,None,Mata Atlântica,PARAIBUNA,SÃO PAULO,SUDESTE
9,1830180379,NOAA-20,2026-08-03 17:11:00,7.9,None,Cerrado,BAMBUÍ,MINAS GERAIS,SUDESTE


## 3. Visualização dos focos de fogo

Clique um ponto para ver os atributos de `municipio`/`estado`.

In [6]:
m.add_gdf(
    focos_in_roi.to_crs("EPSG:4326"),
    layer_name="Focos de fogo",
    info_mode="on_click",
)
m


Map(bottom=9479.0, center=[-21.191064142619037, -47.64885220711009], controls=(ZoomControl(options=['position'…